"""
=============================================================================
## A3 — Block Redundancy Analysis (Marginal Pair Contribution) 
### eMPI Blocking Scheme Feasibility Assessment — Track A, Workstream 3
=============================================================================

PURPOSE:
    For each of the 9 blocking schemes, compute the EXACT number of
    candidate pairs that block contributes which no other block already
    captures. This is the "marginal contribution" of each block.

    The question being answered:
        "If we removed block X from the scheme, how many candidate pairs
         would we lose that no other block catches?"

    A block with high marginal contribution earns its place in the scheme.
    A block with near-zero marginal contribution is fully redundant with
    other blocks and may be a candidate for removal.

WHY THIS MATTERS:
    The Methodology section of the research paper must justify each
    blocking scheme's inclusion with evidence. A3 produces that evidence
    as a hard number: "Block B9 contributes 1,247 unique pairs not
    captured by any other block."

HOW TO USE:
    Run cells sequentially top to bottom. The script assumes the
    cleaned dataset is available with all clean_* columns and EDA flag
    columns from your team's cleaning pipeline.

INPUT REQUIREMENTS:
    A pandas DataFrame called `df_clean` with the following columns:
      - PATID                  (unique row identifier)
      - clean_LastNM           (cleaned last name)
      - clean_FirstNM          (cleaned first name)
      - clean_BirthDT          (parsed datetime or date)
      - clean_SSN              (validated 9-digit SSN, null if invalid)
      - clean_ZipCD            (5-digit ZIP, null if invalid)
      - clean_Email            (validated lowercase email, null if invalid)
      - Consolidated_Phone_Array  (list column of cleaned phones per record)
      - dm_LastNM              (Double Metaphone primary code of LastNM)
      - soundex_LastNM         (Soundex code of LastNM)
      - soundex_FirstNM        (Soundex code of FirstNM)

    If column names differ in your environment, update the constants in
    Cell 1 below.

OUTPUT:
    1. a3_results: dict containing per-block marginal contribution stats
    2. a3_summary: pandas DataFrame ready for the report
    3. a3_pair_audit: detailed table showing block coverage per pair
    4. CSV exports: a3_marginal_contribution.csv, a3_pair_audit_sample.csv
    5. Visualization: a3_marginal_contribution_chart.png
    6. HTML report: a3_redundancy_report.html

## HIPAA NOTE:
    This script never prints raw blocking key values (which contain PHI).
    All console output is aggregate statistics only. The pair audit table
    references only PATID values, which are non-PHI synthetic identifiers.
=============================================================================
"""


In [ ]:
%pip install matplotlib

In [ ]:
# =============================================================================
# CELL 1 — Imports and Configuration
# =============================================================================

import warnings
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


warnings.filterwarnings("ignore")

# ── Input dataframe ─────────────────────────────────────────────────────────
# REPLACE THIS LINE WITH YOUR CLEANED DATAFRAME LOAD
df_clean = pd.read_parquet(r"C:\Users\Public\Documents\EMPI_latest\data\processed\MDM_Population_cleaned_v1_2026_05_24.parquet")
# For now, this assumes df_clean is already loaded in the notebook

TOTAL_RECORDS = len(df_clean)
print(f"✓ A3 Redundancy Analysis starting")
print(f"✓ Cleaned dataset has {TOTAL_RECORDS:,} records")
print(f"✓ Analysis will compute marginal contribution for all 9 blocks\n")



In [ ]:
# ── Diagnostic: print all column names in df_clean ──
print("Columns in df_clean:")
for col in sorted(df_clean.columns.tolist()):
    print(f"  {col}")

In [ ]:

# ── Column name mappings ─────────────────────────────────────────────────────
# Confirmed against df_clean column list — May 2026

COL_PATID         = "PATID"
COL_LAST_NM       = "LastNM_clean"
COL_FIRST_NM      = "FirstNM_clean"
COL_BIRTH_DT      = "BirthDT_clean"
COL_SSN           = "SSN_clean"
COL_ZIP           = "ZipCD_clean_base"
COL_EMAIL         = "Email_clean"
COL_PHONE_ARRAY   = "Phones_set"

# Phonetic encoding column names — computed inline, do not change
COL_DM_LASTNM     = "dm_LastNM"
COL_SX_LASTNM     = "soundex_LastNM"
COL_SX_FIRSTNM    = "soundex_FirstNM"

In [ ]:
%pip install jellyfish phonetics

In [ ]:
# =============================================================================
# CELL 2 — Compute Derived Columns Needed for Blocking Keys
# =============================================================================

print("── Computing derived blocking key columns ──────────────────────────")

import jellyfish
import phonetics

work = df_clean.copy()

# ── Date of birth derived columns ──────────────────────────────────────────
work["_dob_str"]    = pd.to_datetime(work[COL_BIRTH_DT], errors="coerce").dt.strftime("%Y-%m-%d")
work["_birth_year"] = pd.to_datetime(work[COL_BIRTH_DT], errors="coerce").dt.year.astype("Int64")
print("  ✓ _dob_str, _birth_year computed")

# ── First name 3-character prefix (B4) ─────────────────────────────────────
work["_fn_prefix3"] = work[COL_FIRST_NM].str[:3]
print("  ✓ _fn_prefix3 computed")

# ── Last 4 of SSN (B9) ─────────────────────────────────────────────────────
work["_ssn_last4"] = work["last_4_SSN"]
print("  ✓ _ssn_last4 mapped from last_4_SSN")

# ── Phonetic encoding helpers ──────────────────────────────────────────────
def _safe_dm_primary(name):
    """Double Metaphone — returns primary code or NaN.
    Uses the 'phonetics' library which is more robute than metaphone package 
    """
    if pd.isna(name) or not isinstance(name, str) or name.strip() == "":
        return np.nan
    try:
        result = phonetics.dmetaphone(name)
        return result[0] if result[0] else np.nan
    except Exception:
        return np.nan

def _safe_soundex(name):
    """Soundex — returns code or NaN."""
    if pd.isna(name) or not isinstance(name, str) or name.strip() == "":
        return np.nan
    try:
        return jellyfish.soundex(name)
    except Exception:
        return np.nan

# ── Compute phonetic encodings on cleaned name columns ─────────────────────
print("  Computing Double Metaphone on clean_LastNM... (30–60 sec)")
work[COL_DM_LASTNM] = work[COL_LAST_NM].apply(_safe_dm_primary)
print(f"    ✓ {COL_DM_LASTNM} computed "
      f"({work[COL_DM_LASTNM].notna().sum():,} non-null values)")

print("  Computing Soundex on clean_LastNM...")
work[COL_SX_LASTNM] = work[COL_LAST_NM].apply(_safe_soundex)
print(f"    ✓ {COL_SX_LASTNM} computed "
      f"({work[COL_SX_LASTNM].notna().sum():,} non-null values)")

print("  Computing Soundex on clean_FirstNM...")
work[COL_SX_FIRSTNM] = work[COL_FIRST_NM].apply(_safe_soundex)
print(f"    ✓ {COL_SX_FIRSTNM} computed "
      f"({work[COL_SX_FIRSTNM].notna().sum():,} non-null values)")

print("\n✓ All derived columns ready for blocking key generation\n")

In [ ]:
# =============================================================================
# CELL 3 — Generate Candidate Pair Sets for Each Block
# =============================================================================
# For each of the 9 blocks, compute the complete set of candidate pairs
# the block would generate. Pairs are stored as (PATID_A, PATID_B)
# tuples with A < B for canonical ordering.

def generate_pairs_from_key(df, key_column, block_name):
    """
    Given a dataframe and a column containing the blocking key value
    for each record, generate all canonical (PATID_A, PATID_B) pairs
    where two records share a non-null key value.

    Returns a set of tuples for fast set operations.
    """
    # Drop records with null key (they don't participate in this block)
    valid = df[[COL_PATID, key_column]].dropna(subset=[key_column])

    # Group records by key value
    grouped = valid.groupby(key_column)[COL_PATID].apply(list)

    pairs = set()
    n_blocks = 0
    n_records_participating = 0

    for patid_list in grouped:
        if len(patid_list) < 2:
            continue  # singleton block — generates no pairs
        n_blocks += 1
        n_records_participating += len(patid_list)
        # Generate all pairs within this block, canonicalized (A < B)
        for a, b in combinations(sorted(patid_list), 2):
            pairs.add((a, b))

    print(f"  {block_name}: {len(pairs):>8,} pairs from "
          f"{n_blocks:>5,} blocks ({n_records_participating:,} records)")
    return pairs


print("── Generating candidate pair sets per block ────────────────────────")

# ── B1: SSN Exact ──
work["_bk_b1"] = work[COL_SSN]
pairs_b1 = generate_pairs_from_key(work, "_bk_b1", "B1 (SSN Exact)         ")

# ── B2: LastNM + Full DOB ──
work["_bk_b2"] = np.where(
    work[COL_LAST_NM].notna() & work["_dob_str"].notna(),
    work[COL_LAST_NM].astype(str) + "|" + work["_dob_str"],
    np.nan
)
pairs_b2 = generate_pairs_from_key(work, "_bk_b2", "B2 (LN + Full DOB)     ")

# ── B3: DM(LastNM) + Full DOB ──
work["_bk_b3"] = np.where(
    work[COL_DM_LASTNM].notna() & work["_dob_str"].notna(),
    work[COL_DM_LASTNM].astype(str) + "|" + work["_dob_str"],
    np.nan
)
pairs_b3 = generate_pairs_from_key(work, "_bk_b3", "B3 (DM-LN + Full DOB)  ")

# ── B4: LastNM + BirthYear + FN Prefix 3 ──
work["_bk_b4"] = np.where(
    work[COL_LAST_NM].notna() & work["_birth_year"].notna() & work["_fn_prefix3"].notna(),
    (work[COL_LAST_NM].astype(str) + "|" +
     work["_birth_year"].astype(str) + "|" +
     work["_fn_prefix3"].astype(str)),
    np.nan
)
pairs_b4 = generate_pairs_from_key(work, "_bk_b4", "B4 (LN + Year + FN3)   ")

# ── B5: Phone Set Intersection (CORRECTED — parses Phones_set strings) ──
import ast

def _parse_phone_set_a3(value):
    """Parse string-serialized Phones_set back to a Python set."""
    if pd.isna(value):
        return set()
    if isinstance(value, (set, list)):
        return set(str(p) for p in value if str(p).strip())
    if not isinstance(value, str):
        return set()
    value = value.strip()
    if value in ("", "nan", "None", "set()", "{}", "[]"):
        return set()
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (set, list, tuple)):
            return set(str(p).strip() for p in parsed if str(p).strip())
    except (ValueError, SyntaxError):
        pass
    try:
        cleaned = value.strip("{}[]").replace("'", "").replace('"', "")
        return {p.strip() for p in cleaned.split(",") if p.strip()}
    except Exception:
        return set()

print(f"  B5 (Phone Set): generating pairs via long-format join...")
phone_long = work[[COL_PATID, COL_PHONE_ARRAY]].copy()
phone_long[COL_PHONE_ARRAY] = phone_long[COL_PHONE_ARRAY].apply(_parse_phone_set_a3)
phone_long = phone_long.explode(COL_PHONE_ARRAY)
phone_long = phone_long[
    phone_long[COL_PHONE_ARRAY].notna() &
    (phone_long[COL_PHONE_ARRAY].astype(str).str.strip() != "")
]
phone_long = phone_long.rename(columns={COL_PHONE_ARRAY: "phone"})
phone_long = phone_long.drop_duplicates(subset=[COL_PATID, "phone"])

phone_pairs_raw = phone_long.merge(phone_long, on="phone", suffixes=("_a", "_b"))
phone_pairs_raw = phone_pairs_raw[
    phone_pairs_raw[f"{COL_PATID}_a"] != phone_pairs_raw[f"{COL_PATID}_b"]
]
pairs_b5 = set()
for a, b in zip(phone_pairs_raw[f"{COL_PATID}_a"],
                phone_pairs_raw[f"{COL_PATID}_b"]):
    pairs_b5.add(tuple(sorted([a, b])))

print(f"  B5 (Phone Set)          : {len(pairs_b5):>8,} pairs "
      f"({phone_long[COL_PATID].nunique():,} records with phones)")

# ── B6: Email Exact ──
work["_bk_b6"] = work[COL_EMAIL]
pairs_b6 = generate_pairs_from_key(work, "_bk_b6", "B6 (Email Exact)       ")

# ── B7: DM(LastNM) + ZIP + BirthYear ──
work["_bk_b7"] = np.where(
    work[COL_DM_LASTNM].notna() & work[COL_ZIP].notna() & work["_birth_year"].notna(),
    (work[COL_DM_LASTNM].astype(str) + "|" +
     work[COL_ZIP].astype(str) + "|" +
     work["_birth_year"].astype(str)),
    np.nan
)
pairs_b7 = generate_pairs_from_key(work, "_bk_b7", "B7 (DM-LN + ZIP + Year)")

# ── B8: Soundex(FN) + Soundex(LN) + BirthYear ──
work["_bk_b8"] = np.where(
    (work[COL_SX_FIRSTNM].notna() & work[COL_SX_LASTNM].notna()
     & work["_birth_year"].notna()),
    (work[COL_SX_FIRSTNM].astype(str) + "|" +
     work[COL_SX_LASTNM].astype(str) + "|" +
     work["_birth_year"].astype(str)),
    np.nan
)
pairs_b8 = generate_pairs_from_key(work, "_bk_b8", "B8 (Sx-FN+Sx-LN+Year)  ")

# ── B9: LastNM + FirstNM + SSN Last 4 ──
work["_bk_b9"] = np.where(
    (work[COL_LAST_NM].notna() & work[COL_FIRST_NM].notna()
     & work["_ssn_last4"].notna()),
    (work[COL_LAST_NM].astype(str) + "|" +
     work[COL_FIRST_NM].astype(str) + "|" +
     work["_ssn_last4"].astype(str)),
    np.nan
)
pairs_b9 = generate_pairs_from_key(work, "_bk_b9", "B9 (LN + FN + SSN_L4)  ")

# Collect all pair sets for downstream analysis
block_pairs = {
    "B1": pairs_b1, "B2": pairs_b2, "B3": pairs_b3,
    "B4": pairs_b4, "B5": pairs_b5, "B6": pairs_b6,
    "B7": pairs_b7, "B8": pairs_b8, "B9": pairs_b9,
}

print(f"\n✓ Pair generation complete for all 9 blocks\n")

In [ ]:
# =============================================================================
# CELL 4 — Compute Union of All Candidate Pairs
# =============================================================================
# The deduplicated union is the actual candidate set that would feed
# into the Fellegi-Sunter scoring engine.

print("── Computing union of candidate pairs across all blocks ─────────────")

all_pairs = set()
for block_id, pairs in block_pairs.items():
    all_pairs |= pairs

total_unique_pairs = len(all_pairs)
total_redundant_pair_count = sum(len(p) for p in block_pairs.values())
naive_all_pairs = TOTAL_RECORDS * (TOTAL_RECORDS - 1) // 2

print(f"  Sum of per-block pairs (with overlap):   {total_redundant_pair_count:>12,}")
print(f"  Unique pairs after union deduplication:  {total_unique_pairs:>12,}")
print(f"  Overlap reduction:                       "
      f"{100 * (1 - total_unique_pairs/total_redundant_pair_count):>11.1f}%")
print(f"")
print(f"  Naive all-pairs comparison space:        {naive_all_pairs:>12,}")
print(f"  Blocking reduction ratio:                "
      f"{100 * (1 - total_unique_pairs/naive_all_pairs):>11.4f}%")
print(f"  Pairs to score in FS engine:             {total_unique_pairs:>12,}\n")



In [ ]:
# =============================================================================
# CELL 5 — Compute Marginal Contribution Per Block
# =============================================================================
# The marginal contribution of block X is the count of pairs in X that
# are NOT in the union of all OTHER blocks.
#
# Formula:
#   marginal(X) = |pairs_X - (pairs_1 ∪ pairs_2 ∪ ... ∪ pairs_n, excluding X)|

print("── Computing marginal contribution per block ───────────────────────")

a3_results = {}

for block_id, pairs_in_block in block_pairs.items():
    # Union of all OTHER blocks (everything except this one)
    other_blocks_union = set()
    for other_id, other_pairs in block_pairs.items():
        if other_id != block_id:
            other_blocks_union |= other_pairs

    # Pairs in this block that no other block catches
    unique_to_block = pairs_in_block - other_blocks_union

    # Pairs in this block also caught by at least one other block
    shared_with_others = pairs_in_block & other_blocks_union

    a3_results[block_id] = {
        "block": block_id,
        "total_pairs": len(pairs_in_block),
        "unique_pairs": len(unique_to_block),
        "shared_pairs": len(shared_with_others),
        "pct_unique": (100 * len(unique_to_block) / len(pairs_in_block)
                       if len(pairs_in_block) > 0 else 0),
    }

    print(f"  {block_id}: "
          f"total={len(pairs_in_block):>7,} | "
          f"unique={len(unique_to_block):>6,} ({a3_results[block_id]['pct_unique']:>5.1f}%) | "
          f"shared={len(shared_with_others):>7,}")

print()


In [ ]:
# =============================================================================
# CELL 6 — Compute Pairwise Overlap Matrix
# =============================================================================
# How much does each block overlap with each other block?
# This produces a 9x9 matrix where cell (i, j) is the count of pairs
# in both block i and block j.

print("── Computing pairwise overlap matrix ───────────────────────────────")

block_ids = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B9"]
overlap_matrix = pd.DataFrame(
    index=block_ids, columns=block_ids, dtype=int
)

for i in block_ids:
    for j in block_ids:
        overlap_matrix.loc[i, j] = len(block_pairs[i] & block_pairs[j])

print("\n  Pairwise Overlap Matrix (cell = # pairs in BOTH blocks):\n")
print(overlap_matrix.to_string())
print()

# ── Compute Jaccard similarity matrix ────────────────────────────────────────
# Jaccard = |intersection| / |union| — a normalized measure of how similar
# two pair sets are. Values close to 1.0 indicate near-complete redundancy.

print("\n  Jaccard Similarity Matrix (1.0 = fully redundant, 0.0 = independent):\n")
jaccard_matrix = pd.DataFrame(
    index=block_ids, columns=block_ids, dtype=float
)
for i in block_ids:
    for j in block_ids:
        intersection = len(block_pairs[i] & block_pairs[j])
        union = len(block_pairs[i] | block_pairs[j])
        jaccard_matrix.loc[i, j] = intersection / union if union > 0 else 0

print(jaccard_matrix.round(3).to_string())
print()



In [ ]:
# =============================================================================
# CELL 7 — Build Pair-Level Audit Table
# =============================================================================
# For each unique candidate pair, record which blocks generated it.
# This is the source-of-truth audit table that the Methodology section
# references when justifying block design decisions.

print("── Building pair-level audit table ─────────────────────────────────")

# For each pair, build a sorted tuple of block IDs that captured it
pair_to_blocks = {}
for block_id, pairs in block_pairs.items():
    for pair in pairs:
        if pair not in pair_to_blocks:
            pair_to_blocks[pair] = []
        pair_to_blocks[pair].append(block_id)

# Convert to dataframe — for memory efficiency on large pair sets,
# we use a sampled version for export and full counts for stats
a3_pair_audit_records = []
for pair, blocks in pair_to_blocks.items():
    a3_pair_audit_records.append({
        "PATID_A": pair[0],
        "PATID_B": pair[1],
        "blocks_capturing": "|".join(sorted(blocks)),
        "n_blocks_capturing": len(blocks),
    })

a3_pair_audit = pd.DataFrame(a3_pair_audit_records)

print(f"  Total unique pairs in audit table: {len(a3_pair_audit):,}")
print(f"\n  Pairs by number of capturing blocks:")
n_blocks_dist = a3_pair_audit["n_blocks_capturing"].value_counts().sort_index()
for n_blocks, count in n_blocks_dist.items():
    pct = 100 * count / len(a3_pair_audit)
    bar = "█" * int(pct / 2)
    print(f"    Captured by {n_blocks} block(s): {count:>8,} pairs ({pct:>5.1f}%) {bar}")

# Save a sample of the audit table for inspection (full table may be too large)
sample_size = min(5000, len(a3_pair_audit))
a3_pair_audit.sample(n=sample_size, random_state=42).to_csv(
    "a3_pair_audit_sample.csv", index=False
)
print(f"\n  ✓ Sample of {sample_size:,} pairs exported to a3_pair_audit_sample.csv\n")

In [ ]:
# =============================================================================
# CELL 8 — Decision Recommendations per Block
# =============================================================================
# Combine marginal contribution + A1 coverage + A2 governance into a
# final recommendation per block.

print("── Generating block recommendations ────────────────────────────────")

# These thresholds determine the recommendation logic. Adjust if your
# team agrees to different criteria.
HIGH_VALUE_PCT_UNIQUE      = 25.0   # >25% unique pairs = high marginal value
MODERATE_VALUE_PCT_UNIQUE  = 10.0   # 10-25% = moderate value
LOW_VALUE_MIN_UNIQUE_PAIRS = 500    # block must contribute ≥500 unique pairs

# Role classification from architecture design
block_roles = {
    "B1": "Precision Anchor",
    "B2": "Recall Workhorse",
    "B3": "Recall Workhorse",
    "B4": "Recall Workhorse",
    "B5": "Recall Supplement",
    "B6": "Precision Anchor",
    "B7": "Recall Supplement",
    "B8": "Recall Supplement",
    "B9": "Precision Anchor",
}

for block_id in block_ids:
    r = a3_results[block_id]
    role = block_roles[block_id]
    pct = r["pct_unique"]
    n_unique = r["unique_pairs"]

    # Decision tree for recommendation
    if n_unique == 0:
        rec = "❌ REMOVE — Zero unique pairs; fully redundant"
    elif pct >= HIGH_VALUE_PCT_UNIQUE:
        rec = "✅ KEEP — High marginal contribution"
    elif pct >= MODERATE_VALUE_PCT_UNIQUE:
        rec = "✅ KEEP — Moderate marginal contribution"
    elif n_unique >= LOW_VALUE_MIN_UNIQUE_PAIRS:
        rec = "✅ KEEP — Targeted contribution"
    else:
        rec = "⚠️  REVIEW — Low marginal contribution; consider removal"

    # Precision anchors are kept regardless of low coverage, as long as
    # they contribute SOME unique pairs (their value is precision, not volume)
    if role == "Precision Anchor" and n_unique > 0 and "REVIEW" in rec:
        rec = "✅ KEEP — Precision Anchor; targeted high-confidence pairs"

    a3_results[block_id]["role"] = role
    a3_results[block_id]["recommendation"] = rec

    print(f"  {block_id} ({role:<19}): {rec}")

print()

In [ ]:
# =============================================================================
# CELL 9 — Build Summary DataFrame
# =============================================================================

a3_summary = pd.DataFrame([
    {
        "Block": r["block"],
        "Role": r["role"],
        "Total Pairs": r["total_pairs"],
        "Unique Pairs": r["unique_pairs"],
        "Shared Pairs": r["shared_pairs"],
        "% Unique": round(r["pct_unique"], 1),
        "Recommendation": r["recommendation"],
    }
    for r in a3_results.values()
]).set_index("Block")

print("── A3 Summary Table ─────────────────────────────────────────────────")
print(a3_summary.to_string())
print()

a3_summary.to_csv("a3_marginal_contribution.csv")
overlap_matrix.to_csv("a3_overlap_matrix.csv")
jaccard_matrix.round(3).to_csv("a3_jaccard_matrix.csv")
print("  ✓ Exported: a3_marginal_contribution.csv")
print("  ✓ Exported: a3_overlap_matrix.csv")
print("  ✓ Exported: a3_jaccard_matrix.csv\n")


In [ ]:
# =============================================================================
# CELL 10 — Visualizations
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(
    "A3 — Block Redundancy Analysis\n"
    "eMPI Blocking Scheme Feasibility Assessment (Track A, Workstream 3)",
    fontsize=14, fontweight="bold", y=0.995
)

# ── Chart 1: Marginal contribution per block (stacked bar) ──
ax1 = axes[0, 0]
x_pos = np.arange(len(block_ids))
unique_counts = [a3_results[b]["unique_pairs"] for b in block_ids]
shared_counts = [a3_results[b]["shared_pairs"] for b in block_ids]

ax1.bar(x_pos, unique_counts, color="#27AE60", label="Unique to this block")
ax1.bar(x_pos, shared_counts, bottom=unique_counts,
        color="#BDC3C7", label="Shared with other blocks")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(block_ids)
ax1.set_ylabel("Number of Candidate Pairs")
ax1.set_title("Total Pairs per Block — Unique vs. Shared", fontweight="bold")
ax1.legend(loc="upper right")
ax1.grid(axis="y", alpha=0.3)

# Annotate each bar with the percentage unique
for i, block_id in enumerate(block_ids):
    total = a3_results[block_id]["total_pairs"]
    pct = a3_results[block_id]["pct_unique"]
    if total > 0:
        ax1.text(i, total + max(unique_counts + shared_counts) * 0.02,
                 f"{pct:.0f}%", ha="center", fontsize=9, fontweight="bold")

# ── Chart 2: Percentage of pairs unique per block ──
ax2 = axes[0, 1]
pct_unique = [a3_results[b]["pct_unique"] for b in block_ids]
colors = ["#27AE60" if p >= 25 else "#F39C12" if p >= 10 else "#E84855"
          for p in pct_unique]
bars = ax2.barh(block_ids, pct_unique, color=colors)
ax2.axvline(x=25, color="green", linestyle="--", alpha=0.5,
            label="High value threshold (25%)")
ax2.axvline(x=10, color="orange", linestyle="--", alpha=0.5,
            label="Moderate value threshold (10%)")
ax2.set_xlabel("% of Block's Pairs Unique to Block")
ax2.set_title("Marginal Contribution % per Block", fontweight="bold")
ax2.legend(loc="lower right", fontsize=8)
ax2.invert_yaxis()
for bar, value in zip(bars, pct_unique):
    ax2.text(value + 1, bar.get_y() + bar.get_height() / 2,
             f"{value:.1f}%", va="center", fontsize=9)

# ── Chart 3: Jaccard similarity heatmap ──
ax3 = axes[1, 0]
im = ax3.imshow(jaccard_matrix.astype(float).values,
                cmap="RdYlGn_r", vmin=0, vmax=1, aspect="auto")
ax3.set_xticks(range(9))
ax3.set_yticks(range(9))
ax3.set_xticklabels(block_ids)
ax3.set_yticklabels(block_ids)
ax3.set_title("Pairwise Jaccard Similarity\n(red = high redundancy, green = independent)",
              fontweight="bold", fontsize=11)
plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
# Annotate cells with values
for i in range(9):
    for j in range(9):
        val = jaccard_matrix.iloc[i, j]
        color = "white" if val > 0.5 else "black"
        ax3.text(j, i, f"{val:.2f}", ha="center", va="center",
                 color=color, fontsize=8)

# ── Chart 4: Pair capture redundancy distribution ──
ax4 = axes[1, 1]
n_blocks_capt = a3_pair_audit["n_blocks_capturing"].value_counts().sort_index()
bars = ax4.bar(n_blocks_capt.index, n_blocks_capt.values, color="#2E86AB")
ax4.set_xlabel("Number of Blocks That Capture Each Pair")
ax4.set_ylabel("Number of Pairs")
ax4.set_title("Pair Capture Redundancy Distribution", fontweight="bold")
ax4.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, n_blocks_capt.values):
    pct = 100 * value / len(a3_pair_audit)
    ax4.text(bar.get_x() + bar.get_width() / 2, value,
             f"{value:,}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("a3_marginal_contribution_chart.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Visualization saved: a3_marginal_contribution_chart.png\n")


In [ ]:
# =============================================================================
# CELL 11 — HTML Report Generation
# =============================================================================

run_timestamp = datetime.now().strftime("%B %d, %Y at %H:%M")

rows_html = ""
for block_id in block_ids:
    r = a3_results[block_id]
    rec_class = (
        "rec-keep" if "KEEP" in r["recommendation"]
        else "rec-warn" if "REVIEW" in r["recommendation"]
        else "rec-remove"
    )

    role_class = ("role-precision" if "Precision" in r["role"]
                  else "role-recall")

    unique_bar_pct = min(int(r["pct_unique"] * 2), 100)

    rows_html += f"""
    <tr>
      <td class="block-id">{block_id}</td>
      <td><span class="role-badge {role_class}">{r['role']}</span></td>
      <td class="num">{r['total_pairs']:,}</td>
      <td class="num strong">{r['unique_pairs']:,}</td>
      <td class="num">{r['shared_pairs']:,}</td>
      <td>
        <div class="bar-wrap">
          <div class="bar" style="width:{unique_bar_pct}%"></div>
          <span class="bar-label">{r['pct_unique']:.1f}%</span>
        </div>
      </td>
      <td class="{rec_class}">{r['recommendation']}</td>
    </tr>"""

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>A3 Redundancy Analysis Report</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', Arial, sans-serif; font-size: 11px;
         color: #1a1a2e; background: #f8f9fa; padding: 20px; }}
  .page {{ background: white; max-width: 1280px; margin: 0 auto;
          padding: 28px 32px; box-shadow: 0 2px 12px rgba(0,0,0,0.1);
          border-radius: 6px; }}
  .header {{ display: flex; justify-content: space-between;
            border-bottom: 3px solid #2E86AB; padding-bottom: 14px;
            margin-bottom: 20px; }}
  .header h1 {{ font-size: 18px; }}
  .header h2 {{ font-size: 12px; color: #2E86AB; margin-top: 3px; }}
  .header-right {{ text-align: right; font-size: 10px; color: #666;
                  line-height: 1.6; }}
  .badge {{ background: #2E86AB; color: white; padding: 3px 10px;
           border-radius: 12px; font-size: 10px; font-weight: 600;
           display: inline-block; margin-bottom: 4px; }}
  .summary-stats {{ display: grid; grid-template-columns: repeat(4, 1fr);
                   gap: 12px; margin-bottom: 18px; }}
  .stat-card {{ background: #f0f7fb; border-left: 4px solid #2E86AB;
               padding: 10px 14px; border-radius: 4px; }}
  .stat-card h3 {{ font-size: 9px; color: #2E86AB; font-weight: 700;
                  text-transform: uppercase; letter-spacing: 0.5px;
                  margin-bottom: 4px; }}
  .stat-card .stat-value {{ font-size: 18px; font-weight: 700;
                            color: #1a1a2e; }}
  .stat-card .stat-context {{ font-size: 9px; color: #666;
                              margin-top: 2px; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 10.5px; }}
  th {{ background: #1a1a2e; color: white; padding: 9px 12px;
        text-align: left; font-size: 10px; font-weight: 600; }}
  td {{ padding: 8px 12px; border-bottom: 1px solid #eee;
        vertical-align: middle; }}
  tr:nth-child(even) td {{ background: #fafbfc; }}
  .block-id {{ font-weight: 700; font-size: 13px; color: #2E86AB; }}
  .num {{ text-align: right; font-variant-numeric: tabular-nums; }}
  .strong {{ font-weight: 700; color: #27AE60; font-size: 11.5px; }}
  .role-badge {{ display: inline-block; padding: 2px 8px;
                border-radius: 10px; font-size: 9px; font-weight: 700;
                text-transform: uppercase; }}
  .role-precision {{ background: #fde8ec; color: #c0392b; }}
  .role-recall {{ background: #e8f5e9; color: #27ae60; }}
  .bar-wrap {{ display: flex; align-items: center; gap: 8px;
              min-width: 120px; }}
  .bar {{ height: 8px; background: #27ae60; border-radius: 4px;
         min-width: 2px; }}
  .bar-label {{ font-size: 10px; font-weight: 600; }}
  .rec-keep {{ color: #27ae60; font-weight: 600; }}
  .rec-warn {{ color: #e67e22; font-weight: 700; }}
  .rec-remove {{ color: #c0392b; font-weight: 700; }}
  .footer {{ margin-top: 18px; border-top: 1px solid #ddd;
            padding-top: 10px; display: flex;
            justify-content: space-between; color: #999; font-size: 9px; }}
</style>
</head>
<body>
<div class="page">
  <div class="header">
    <div>
      <h1>A3 — Block Redundancy Analysis</h1>
      <h2>eMPI Blocking Scheme Feasibility Assessment</h2>
    </div>
    <div class="header-right">
      <div class="badge">INTERNAL — DRAFT</div><br>
      Generated: {run_timestamp}<br>
      Dataset: {TOTAL_RECORDS:,} cleaned records<br>
      Total unique candidate pairs: {total_unique_pairs:,}
    </div>
  </div>

  <div class="summary-stats">
    <div class="stat-card">
      <h3>Total Unique Pairs</h3>
      <div class="stat-value">{total_unique_pairs:,}</div>
      <div class="stat-context">After union deduplication</div>
    </div>
    <div class="stat-card">
      <h3>Blocking Reduction</h3>
      <div class="stat-value">{100 * (1 - total_unique_pairs/naive_all_pairs):.3f}%</div>
      <div class="stat-context">From {naive_all_pairs:,} naive pairs</div>
    </div>
    <div class="stat-card">
      <h3>Overlap Across Blocks</h3>
      <div class="stat-value">{100 * (1 - total_unique_pairs/total_redundant_pair_count):.1f}%</div>
      <div class="stat-context">Redundancy in candidate generation</div>
    </div>
    <div class="stat-card">
      <h3>Blocks Recommended</h3>
      <div class="stat-value">
        {sum(1 for r in a3_results.values() if 'KEEP' in r['recommendation'])} / 9
      </div>
      <div class="stat-context">For inclusion in final scheme</div>
    </div>
  </div>

  <table>
    <thead>
      <tr>
        <th>Block</th>
        <th>Role</th>
        <th>Total Pairs</th>
        <th>Unique Pairs</th>
        <th>Shared Pairs</th>
        <th>% Unique</th>
        <th>Recommendation</th>
      </tr>
    </thead>
    <tbody>
      {rows_html}
    </tbody>
  </table>

  <div class="footer">
    <span>eMPI Capstone — AllianceChicago | Methodology Section Evidence Base</span>
    <span>A3 Owner: Analyst | Generated: {run_timestamp}</span>
  </div>
</div>
</body>
</html>"""

with open("a3_redundancy_report.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✅ A3 Redundancy Analysis Complete")
print(f"   ✓ Summary table:  a3_marginal_contribution.csv")
print(f"   ✓ Overlap matrix: a3_overlap_matrix.csv")
print(f"   ✓ Jaccard matrix: a3_jaccard_matrix.csv")
print(f"   ✓ Pair audit:     a3_pair_audit_sample.csv")
print(f"   ✓ Chart:          a3_marginal_contribution_chart.png")
print(f"   ✓ HTML report:    a3_redundancy_report.html")
